# NB01 — Discovery and Sample Design

**Project**: NEON metagenome lineage novelty and functional ecology

## Goal

Reproduce the planning-session discovery queries that bounded the project scope, and emit canonical inventory artifacts:

- `data/sample_inventory.tsv` — one row per NEON biosample with habitat, chemistry, coords, depth, collection date
- `data/workflow_inventory.tsv` — one row per workflow run with biosample/study links
- `data/mag_inventory.tsv` — one row per MAG with GTDB taxonomy, completeness, contamination, bin_quality, habitat, lowest-defined rank

Runs against the BERDL JupyterHub Spark session.

In [1]:
import os
import re
import pandas as pd
from pyspark.sql import functions as F

spark = get_spark_session()

STUDIES = {
    'nmdc:sty-11-34xj1150': 'soil',
    'nmdc:sty-11-hht5sb92': 'water',
    'nmdc:sty-11-pzmd0x14': 'benthic',
}
STUDY_IDS = list(STUDIES.keys())
OUT_DIR = os.path.abspath('../data')
os.makedirs(OUT_DIR, exist_ok=True)
studies_sql = ', '.join(f"'{s}'" for s in STUDY_IDS)

## 1. Confirm the three NEON studies

In [2]:
studies = spark.sql(f"""
    SELECT id, title, principal_investigator_name
    FROM nmdc_metadata.study_set
    WHERE id IN ({studies_sql})
    ORDER BY id
""").toPandas()
studies

,id,title,principal_investigator_name
0,nmdc:sty-11-34xj1150,National Ecological Observatory Network: soil ...,Kate Thibault
1,nmdc:sty-11-hht5sb92,National Ecological Observatory Network: surfa...,Kate Thibault
2,nmdc:sty-11-pzmd0x14,National Ecological Observatory Network: benth...,Kate Thibault


## 2. Sample inventory

Project explicit columns — `biosample_set` has 1,398. Field-name flavors verified against the live schema:

- Scalar columns: `ph`, `elev`, `soil_horizon` (string), `water_content` (array<string>)
- `_has_numeric_value` form: temp, depth, org_carb, tot_nitro, ammonium_nitrogen, carb_nitro_ratio, nitro, conduc, diss_oxygen, samp_size
- `_has_raw_value` form: collection_date, geo_loc_name, env_package
- Flat lat/lon strings: `lat_lon_latitude`, `lat_lon_longitude`

In [3]:
BIOSAMPLE_COLS = [
    'id', 'name', 'collection_date_has_raw_value',
    'lat_lon_latitude', 'lat_lon_longitude',
    'elev',
    'depth_has_numeric_value', 'depth_has_minimum_numeric_value', 'depth_has_maximum_numeric_value',
    'geo_loc_name_has_raw_value',
    'env_broad_scale_term_id', 'env_broad_scale_term_name',
    'env_local_scale_term_id', 'env_local_scale_term_name',
    'env_medium_term_id', 'env_medium_term_name',
    'env_package_has_raw_value',
    'soil_horizon',
    'ph',
    'temp_has_numeric_value',
    'org_carb_has_numeric_value', 'tot_nitro_content_has_numeric_value',
    'ammonium_nitrogen_has_numeric_value', 'carb_nitro_ratio_has_numeric_value',
    'nitro_has_numeric_value',
    'conduc_has_numeric_value', 'diss_oxygen_has_numeric_value',
    'samp_size_has_numeric_value',
    'ecosystem', 'ecosystem_type', 'specific_ecosystem',
]
cols_sql = ', '.join(f'bs.`{c}`' for c in BIOSAMPLE_COLS)

samples = spark.sql(f"""
    SELECT bsa.associated_studies AS study_id, {cols_sql},
           bs.water_content AS water_content_raw
    FROM nmdc_metadata.biosample_set bs
    JOIN nmdc_metadata.biosample_set_associated_studies bsa
      ON bs.id = bsa.parent_id
    WHERE bsa.associated_studies IN ({studies_sql})
""").toPandas()
samples['habitat'] = samples['study_id'].map(STUDIES)

# water_content is array<string>; parse the first numeric out
def _wc(v):
    if v is None: return None
    if isinstance(v, (list, tuple)) and len(v):
        m = re.search(r'-?\d+\.?\d*', str(v[0]))
        return float(m.group()) if m else None
    return None
samples['water_content_pct'] = samples['water_content_raw'].apply(_wc)
samples = samples.drop(columns=['water_content_raw'])
samples.shape

(7459, 34)

In [4]:
samples.groupby('habitat').size()

habitat
benthic     736
soil       6489
water       234
dtype: int64

In [5]:
samples.to_csv(f'{OUT_DIR}/sample_inventory.tsv', sep='\t', index=False)
print(f'wrote sample_inventory.tsv ({len(samples)} rows)')

wrote sample_inventory.tsv (7459 rows)


## 3. Workflow inventory

Resolve study → data_generation → workflow_execution. `was_informed_by` is an array, hence `array_contains`.

In [6]:
workflows = spark.sql(f"""
    SELECT dga.associated_studies AS study_id,
           dga.parent_id AS data_generation_id,
           we.id AS workflow_id,
           we.type AS workflow_type
    FROM nmdc_metadata.data_generation_set_associated_studies dga
    JOIN nmdc_metadata.workflow_execution_set we
      ON array_contains(we.was_informed_by, dga.parent_id)
    WHERE dga.associated_studies IN ({studies_sql})
""").toPandas()
workflows['habitat'] = workflows['study_id'].map(STUDIES)
workflows.groupby(['habitat','workflow_type']).size().unstack(fill_value=0)

workflow_type,nmdc:MagsAnalysis,nmdc:MetagenomeAnnotation,nmdc:MetagenomeAssembly,nmdc:ReadBasedTaxonomyAnalysis,nmdc:ReadQcAnalysis
habitat,,,,,
benthic,594,710,594,600,600
soil,1566,2266,2052,2069,2206
water,197,199,200,201,201


In [7]:
workflows.to_csv(f'{OUT_DIR}/workflow_inventory.tsv', sep='\t', index=False)
print(f'wrote workflow_inventory.tsv ({len(workflows)} rows)')

wrote workflow_inventory.tsv (14255 rows)


## 4. MAG inventory

From `workflow_execution_set_mags_list`. Join via `parent_id` = workflow id.

In [8]:
mags = spark.sql(f"""
    SELECT dga.associated_studies AS study_id,
           dga.parent_id AS data_generation_id,
           we.id AS workflow_id,
           m.bin_name, m.bin_quality,
           m.completeness, m.contamination,
           m.gene_count, m.number_of_contig, m.total_bases,
           m.gtdbtk_domain, m.gtdbtk_phylum, m.gtdbtk_class, m.gtdbtk_order,
           m.gtdbtk_family, m.gtdbtk_genus, m.gtdbtk_species,
           m.eukaryotic_evaluation_completeness, m.eukaryotic_evaluation_contamination,
           m.eukaryotic_evaluation_ncbi_lineage,
           m.num_16s, m.num_23s, m.num_5s, m.num_t_rna
    FROM nmdc_metadata.data_generation_set_associated_studies dga
    JOIN nmdc_metadata.workflow_execution_set we
      ON array_contains(we.was_informed_by, dga.parent_id)
    JOIN nmdc_metadata.workflow_execution_set_mags_list m
      ON m.parent_id = we.id
    WHERE dga.associated_studies IN ({studies_sql})
""").toPandas()
mags['habitat'] = mags['study_id'].map(STUDIES)
print('total MAGs:', len(mags))
mags.groupby(['habitat','bin_quality']).size().unstack(fill_value=0)

total MAGs: 7298


bin_quality,HQ,LQ,MQ
habitat,,,
benthic,21,1209,301
soil,110,4032,1067
water,2,448,108


In [9]:
# Add a lowest-defined-rank column for NB02 lineage novelty analysis
rank_order = ['gtdbtk_species','gtdbtk_genus','gtdbtk_family','gtdbtk_order','gtdbtk_class','gtdbtk_phylum','gtdbtk_domain']
def lowest_rank(row):
    for r in rank_order:
        v = row[r]
        if isinstance(v, str) and v.strip():
            return r.replace('gtdbtk_','')
    return 'unclassified'
mags['lowest_rank'] = mags.apply(lowest_rank, axis=1)
mags.groupby(['habitat','lowest_rank']).size().unstack(fill_value=0)

lowest_rank,species
habitat,
benthic,1531
soil,5209
water,558


In [10]:
mags.to_csv(f'{OUT_DIR}/mag_inventory.tsv', sep='\t', index=False)
print(f'wrote mag_inventory.tsv ({len(mags)} rows)')

wrote mag_inventory.tsv (7298 rows)


## 5. Sanity check against discovery probe

Counts should match `data/discovery_summary.md` (within small drift from the live tenant since the probe date).

In [11]:
expected = {
    'soil':    {'biosamples': 6489, 'mq_hq_mags': 1177},
    'water':   {'biosamples':  234, 'mq_hq_mags':  110},
    'benthic': {'biosamples':  736, 'mq_hq_mags':  322},
}
observed = {h: {'biosamples': int(samples[samples.habitat==h].shape[0]),
                'mq_hq_mags': int(mags[(mags.habitat==h) & (mags.bin_quality.isin(['MQ','HQ']))].shape[0])}
            for h in expected}
pd.DataFrame({'expected': expected, 'observed': observed})

,expected,observed
soil,"{'biosamples': 6489, 'mq_hq_mags': 1177}","{'biosamples': 6489, 'mq_hq_mags': 1177}"
water,"{'biosamples': 234, 'mq_hq_mags': 110}","{'biosamples': 234, 'mq_hq_mags': 110}"
benthic,"{'biosamples': 736, 'mq_hq_mags': 322}","{'biosamples': 736, 'mq_hq_mags': 322}"
